* JEPA


#### Goals
- context encoder, target encoder, predictor가 각각 무엇을 보는지 확인
- loss가 입력 복원이 아니라 embedding 거리라는 점 확인
- target encoder가 student의 EMA로만 갱신되는지 확인


#### Contents
1. Implementation
    <br>1.1 세 모듈
    <br>1.2 mask: predictor는 target 값을 받지 않음
2. Experiment
    <br>2.1 짧은 학습


## 1. Implementation

JEPA(Joint-Embedding Predictive Architecture)는 보이지 않는 구간의 representation을, 보이는 구간의 representation으로 예측한다.

- context encoder: 보이는 토큰만 인코딩
- target encoder: 전체 토큰을 인코딩하고, gradient는 끊는다. 가중치는 context encoder의 EMA
- predictor: context embedding과 target의 위치만 받는다. target 토큰 값은 받지 않는다
- loss: 예측 embedding과 target embedding의 Smooth L1

I-JEPA와 Vol-JEPA는 이 뼈대에서 tokenizer와 mask만 바꾼 것이다.


#### 1.1 세 모듈


In [1]:
import sys
from pathlib import Path

import torch

candidates = [Path.cwd(), Path.cwd() / "jepa"]
for cand in candidates:
    module_dir = cand / "modules"
    if module_dir.is_dir():
        sys.path.insert(0, str(module_dir))
        break
else:
    raise FileNotFoundError("jepa/modules 를 찾지 못했다. 노트북 cwd 를 ai-lab 또는 ai-lab/jepa 로 둔다.")

torch.manual_seed(0)
device = torch.device("cpu")
print("device", device)

from jepa import JEPA

model = JEPA(token_dim=32, embed_dim=64, depth=2, num_heads=4, predictor_depth=1).to(device)
tokens = torch.randn(4, 16, 32, device=device)
n_train = sum(p.numel() for p in model.trainable_parameters())
n_target = sum(p.numel() for p in model.target_encoder.parameters())
print(f"trainable {n_train/1e6:.3f}M, target encoder {n_target/1e6:.3f}M, target requires_grad {any(p.requires_grad for p in model.target_encoder.parameters())}")


device cpu
trainable 0.161M, target encoder 0.102M, target requires_grad False


#### 1.2 mask

target은 연속 구간 하나다. context는 그 바깥이다. predictor 안의 mask token에는 위치 인코딩만 더해진다.


In [2]:
context_idx, target_idx = model.sample_mask(tokens.size(1), target_len=4, device=tokens.device)
print("context", context_idx.tolist())
print("target", target_idx.tolist())
loss = model(tokens, target_len=4)
print("embedding loss", round(loss.item(), 4))


context [0, 1, 2, 3, 4, 9, 10, 11, 12, 13, 14, 15]
target [5, 6, 7, 8]
embedding loss 0.4336


## 2. Experiment

#### 2.1 짧은 학습

입력은 합성 토큰이다. 손실이 유한하고 target encoder 가중치가 student를 천천히 따라가는지만 본다. ImageNet 학습이 아니다.


In [3]:
optimizer = torch.optim.AdamW(model.trainable_parameters(), lr=1e-3, weight_decay=0.05)
history = []
for step in range(8):
    optimizer.zero_grad()
    loss = model(tokens, target_len=4)
    loss.backward()
    optimizer.step()
    model.update_target(momentum=0.996)
    history.append(loss.item())
print([round(v, 4) for v in history])


[0.4313, 0.371, 0.3224, 0.3112, 0.318, 0.2532, 0.305, 0.2222]
